In [2]:
import pandas as pd
import re
import os
from os.path import join
import numpy as np

base_path = "/home/aici/PycharmProjects/survival_analysis/results/attr/ml"
folder = "20_02_2026_brier_best_hyperparams"
out_folder = join(base_path, folder, "plots_final")
os.makedirs(out_folder, exist_ok=True)
res = pd.read_excel(join(base_path, folder, "results_table.xlsx"), sheet_name="Sheet2")

In [3]:
splits = {
    "val": "Internal Validation",
    "swiss": "Swiss Test",
    "vienna": "Vienna Test",
    "swiss_vienna_treated": "Test sets combined (treated)",
}

# --- metrics ---
metrics = {
    "harrell": "Harrell",
    "uno": "Uno",
    "ant": "Antolini",
}
metrics_map = {
    "ici": "ICI",
    "mCalib": "Mean Calibration"
}

In [4]:
def split_metric_name(col_name):
    """
    Convert single-level metric name to (family, label) tuple
    """
    # C-index metrics
    if col_name.startswith(("Harrell", "Uno", "Antolini")):
        return ("C-index", col_name.replace(" (95% CI)", ""))
    # AUC metrics
    m = re.match(r"AUC \((\d+)y\)", col_name)
    if m:
        year = int(m.group(1))
        return ("AUC", f"{year} Year")
    # ICI metrics
    m = re.match(r"ICI \((\d+)y\)", col_name)
    if m:
        year = int(m.group(1))
        return ("ICI", f"{year} Year")
    # Mean Calibration metrics
    m = re.match(r"Mean Calibration \((\d+)y\)", col_name)
    if m:
        year = int(m.group(1))
        return ("Mean Calibration", f"{year} Year")
    # fallback
    return ("Other", col_name)

def round_ci(s):
    down, up = s.replace("]", "").replace("[", "").replace(" ", "").split(",")
    s = f"[{np.round(float(down), 2)}, {np.round(float(up), 2)}]"
    return s

def get_cindex_table(data):
    data = data.copy()
    # --- evaluation sets / splits ---
    auc_times = ["1y", "2y", "3y"]
    for t in auc_times:
        metrics[f"auc_{t}"] = f"AUC ({t})"

    id_vars = ["Model"]

    # --- columns to extract and map ---
    value_cols = []
    metric_map = {}

    for split in splits:
        for m, m_name in metrics.items():
            val_col = f"{m}_{split}"
            ci_col = f"CI_{m}_{split}"
            data[ci_col] = data.apply(lambda x: f"{x[val_col]} {round_ci(x[ci_col])}", axis=1)
            value_cols.extend([val_col, ci_col])
            metric_map[val_col] = m_name
            metric_map[ci_col] = f"{m_name} (95% CI)"

    columns = id_vars + value_cols

    # --- melt ---
    df_long = data[columns].melt(
        id_vars=id_vars,
        value_vars=value_cols,
        var_name="Metric_Source",
        value_name="Value",
    )

    # --- extract Set ---
    pattern = "(" + "|".join(splits) + ")$"
    df_long["Set"] = df_long["Metric_Source"].str.replace("^CI_", "", regex=True).str.extract(pattern)[0]
    df_long["Set"] = df_long["Set"].map(splits).fillna(df_long["Set"])

    # --- map metric names ---
    df_long["Metric"] = df_long["Metric_Source"].map(metric_map)

    # --- ordering ---
    metric_order = [f"{name} (95% CI)" for name in ["Harrell", "Uno", "Antolini"]] + \
                   [f"AUC ({t}) (95% CI)" for t in auc_times]

    df_long["Metric"] = pd.Categorical(df_long["Metric"], categories=metric_order, ordered=True)
    df_long["Set"] = pd.Categorical(
        df_long["Set"],
        categories=list(splits.values()),
        ordered=True,
    )
    # --- pivot ---
    table = df_long.pivot_table(
        index=["Model", "Set"],
        columns=["Metric"],
        values="Value",
        aggfunc="first",
        observed=False,
    )
    # current columns
    cols = table.columns.tolist()

    # convert to list of tuples (level0, level1)
    multi_cols = [split_metric_name(c) for c in cols]

    # set MultiIndex
    table.columns = pd.MultiIndex.from_tuples(multi_cols, names=["Metric Family", "Metric"])
    return table

In [105]:
table = get_cindex_table(res)
table = table.loc[["RSF", "NAC"]]
table.to_excel(join(out_folder, "discrimination_table.xlsx"))

In [106]:
def round_ci(s):
    down, up = s.replace("]", "").replace("[", "").replace(" ", "").split(",")
    s = f"[{np.round(float(down), 2)}, {np.round(float(up), 2)}]"
    return s

In [107]:
data = res.copy()

auc_times = ["1y", "2y", "3y"]
for t in auc_times:
    metrics[f"auc_{t}"] = f"AUC ({t})"

id_vars = ["Model"]

# --- columns to extract and map ---
value_cols = []
metric_map = {}

for split in splits:
    for m, m_name in metrics.items():
        val_col = f"{m}_{split}"
        ci_col = f"CI_{m}_{split}"
        data[val_col] = data.apply(lambda x: f"{np.round(x[val_col], 2)}\n{round_ci(x[ci_col])}", axis=1)
        value_cols.extend([val_col, ci_col])
        metric_map[val_col] = m_name
        metric_map[ci_col] = f"{m_name}"

columns = id_vars + value_cols

# --- melt ---
df_long = data[columns].melt(
    id_vars=id_vars,
    value_vars=value_cols,
    var_name="Metric_Source",
    value_name="Value",
)

# --- extract Set ---
pattern = "(" + "|".join(splits) + ")$"
df_long["Set"] = df_long["Metric_Source"].str.replace("^CI_", "", regex=True).str.extract(pattern)[0]
df_long["Set"] = df_long["Set"].map(splits).fillna(df_long["Set"])

# --- map metric names ---
df_long["Metric"] = df_long["Metric_Source"].map(metric_map)
df_long["est_ci"] = df_long["Metric_Source"].apply(lambda x: "Estimate" if "CI" not in x else "95% CI")

# --- ordering ---
metric_order = [f"{name}" for name in ["Harrell", "Uno", "Antolini"]] + \
               [f"AUC ({t})" for t in auc_times]

df_long["Metric"] = pd.Categorical(df_long["Metric"], categories=metric_order, ordered=True)
df_long["Set"] = pd.Categorical(
    df_long["Set"],
    categories=list(splits.values()),
    ordered=True,
)
# --- pivot ---
table = df_long.pivot_table(
    index=["Metric"],
    columns=["Set", "Model"],
    values="Value",
    aggfunc="first",
    observed=False,
)
# Change index order
cols = table.index.tolist()
multi_cols = [split_metric_name(c) for c in cols]
table.index = pd.MultiIndex.from_tuples(multi_cols, names=["Metric Family", "Metric"])

# Change columns order
set_order = ['Internal Validation', 'Swiss Test', 'Vienna Test', 'Test sets combined (treated)']
model_order = ['RSF', 'NAC']
table = table.reindex(
    columns=sorted(
        table.columns,
        key=lambda x: (set_order.index(x[0]), model_order.index(x[1]))
    )
)

In [108]:
table.to_excel(join(out_folder, "discrimination_table_v2.xlsx"))
table

Set                    Internal Validation                      \
Model                                  RSF                 NAC   
Metric Family Metric                                             
C-index       Harrell    0.75\n[0.7, 0.78]  0.64\n[0.59, 0.68]   
              Uno       0.72\n[0.67, 0.76]  0.61\n[0.57, 0.65]   
              Antolini  0.74\n[0.69, 0.78]  0.63\n[0.57, 0.68]   
AUC           1 Year     0.77\n[0.7, 0.84]  0.67\n[0.56, 0.76]   
              2 Year    0.75\n[0.68, 0.81]  0.61\n[0.52, 0.68]   
              3 Year     0.76\n[0.7, 0.83]  0.62\n[0.54, 0.69]   

Set                             Swiss Test                      \
Model                                  RSF                 NAC   
Metric Family Metric                                             
C-index       Harrell   0.79\n[0.72, 0.85]  0.66\n[0.57, 0.73]   
              Uno       0.79\n[0.73, 0.85]  0.64\n[0.57, 0.71]   
              Antolini   0.77\n[0.7, 0.84]  0.66\n[0.58, 0.74]   
AUC           1 Year    0.79\n[0.69, 0.88]  0.68\n[0.57, 0.78]   
              2 Year    0.77\n[0.68, 0.85]  0.64\n[0.55, 0.73]   
              3 Year    0.83\n[0.74, 0.91]  0.66\n[0.58, 0.75]   

Set                            Vienna Test                      \
Model                                  RSF                 NAC   
Metric Family Metric                                             
C-index       Harrell   0.72\n[0.68, 0.77]  0.66\n[0.61, 0.71]   
              Uno       0.71\n[0.66, 0.78]   0.64\n[0.58, 0.7]   
              Antolini  0.73\n[0.68, 0.78]  0.66\n[0.61, 0.71]   
AUC           1 Year    0.73\n[0.63, 0.82]     0.7\n[0.6, 0.8]   
              2 Year    0.75\n[0.67, 0.83]  0.68\n[0.59, 0.76]   
              3 Year    0.79\n[0.71, 0.86]   0.7\n[0.62, 0.77]   

Set                    Test sets combined (treated)                      
Model                                           RSF                 NAC  
Metric Family Metric                                                     
C-index       Harrell              0.75\n[0.7, 0.8]  0.68\n[0.62, 0.75]  
              Uno                0.76\n[0.71, 0.81]   0.7\n[0.64, 0.75]  
              Antolini            0.74\n[0.69, 0.8]  0.68\n[0.62, 0.75]  
AUC           1 Year             0.75\n[0.66, 0.83]    0.71\n[0.6, 0.8]  
              2 Year             0.74\n[0.66, 0.81]  0.67\n[0.58, 0.75]  
              3 Year             0.81\n[0.73, 0.87]  0.71\n[0.63, 0.78]

In [109]:
data = res.copy()

auc_times = ["1y", "2y", "3y"]
for t in auc_times:
    metrics[f"auc_{t}"] = f"AUC ({t})"

id_vars = ["Model"]

# --- columns to extract and map ---
value_cols = []
metric_map = {}

for split in splits:
    for m, m_name in metrics.items():
        val_col = f"{m}_{split}"
        ci_col = f"CI_{m}_{split}"
        data[val_col] = data.apply(lambda x: f"{np.round(x[val_col], 2)}", axis=1)
        data[ci_col] = data.apply(lambda x: f"{round_ci(x[ci_col])}", axis=1)
        value_cols.extend([val_col, ci_col])
        metric_map[val_col] = m_name
        metric_map[ci_col] = f"{m_name}"

columns = id_vars + value_cols

# --- melt ---
df_long = data[columns].melt(
    id_vars=id_vars,
    value_vars=value_cols,
    var_name="Metric_Source",
    value_name="Value",
)

# --- extract Set ---
pattern = "(" + "|".join(splits) + ")$"
df_long["Set"] = df_long["Metric_Source"].str.replace("^CI_", "", regex=True).str.extract(pattern)[0]
df_long["Set"] = df_long["Set"].map(splits).fillna(df_long["Set"])

# --- map metric names ---
df_long["Metric"] = df_long["Metric_Source"].map(metric_map)
df_long["est_ci"] = df_long["Metric_Source"].apply(lambda x: "Estimate" if "CI" not in x else "95% CI")

# --- ordering ---
metric_order = [f"{name}" for name in ["Harrell", "Uno", "Antolini"]] + \
               [f"AUC ({t})" for t in auc_times]

df_long["Metric"] = pd.Categorical(df_long["Metric"], categories=metric_order, ordered=True)
df_long["Set"] = pd.Categorical(
    df_long["Set"],
    categories=list(splits.values()),
    ordered=True,
)
# --- pivot ---
table = df_long.pivot_table(
    index=["Metric", "est_ci"],
    columns=["Set", "Model"],
    values="Value",
    aggfunc="first",
    observed=False,
)
# Change index order
cols = table.index.tolist()
multi_cols = [split_metric_name(c[0]) + (c[1],) for c in cols]
table.index = pd.MultiIndex.from_tuples(multi_cols, names=["Metric Family", "", "Est_CI"])

# Define desired orders
metric_family_order = ['C-index', 'AUC']
c_index_order = ['Harrell', 'Uno', 'Antolini']
auc_order = ['1 Year', '2 Year', '3 Year']
est_ci_order = ['Estimate', '95% CI']

# Function to get sort key for each tuple
def sort_key(t):
    metric, sub, est_ci = t
    metric_idx = metric_family_order.index(metric)

    if metric == 'C-index':
        sub_idx = c_index_order.index(sub)
    else:  # AUC
        sub_idx = auc_order.index(sub)

    est_ci_idx = est_ci_order.index(est_ci)

    return (metric_idx, sub_idx, est_ci_idx)

# Reorder MultiIndex
new_index = pd.MultiIndex.from_tuples(sorted(table.index, key=sort_key), names=table.index.names)

# Change columns order
model_order = ['RSF', 'NAC']
table = table.reindex(
    columns=sorted(
        table.columns,
        key=lambda x: (set_order.index(x[0]), model_order.index(x[1]))
    )
)

# Reindex DataFrame
table = table.reindex(new_index)

In [110]:
table.to_excel(join(out_folder, "discrimination_table_v3.xlsx"))
table

Set                             Internal Validation                \
Model                                           RSF           NAC   
Metric Family          Est_CI                                       
C-index       Harrell  Estimate                0.75          0.64   
                       95% CI           [0.7, 0.78]  [0.59, 0.68]   
              Uno      Estimate                0.72          0.61   
                       95% CI          [0.67, 0.76]  [0.57, 0.65]   
              Antolini Estimate                0.74          0.63   
                       95% CI          [0.69, 0.78]  [0.57, 0.68]   
AUC           1 Year   Estimate                0.77          0.67   
                       95% CI           [0.7, 0.84]  [0.56, 0.76]   
              2 Year   Estimate                0.75          0.61   
                       95% CI          [0.68, 0.81]  [0.52, 0.68]   
              3 Year   Estimate                0.76          0.62   
                       95% CI           [0.7, 0.83]  [0.54, 0.69]   

Set                                Swiss Test                 Vienna Test  \
Model                                     RSF           NAC           RSF   
Metric Family          Est_CI                                               
C-index       Harrell  Estimate          0.79          0.66          0.72   
                       95% CI    [0.72, 0.85]  [0.57, 0.73]  [0.68, 0.77]   
              Uno      Estimate          0.79          0.64          0.71   
                       95% CI    [0.73, 0.85]  [0.57, 0.71]  [0.66, 0.78]   
              Antolini Estimate          0.77          0.66          0.73   
                       95% CI     [0.7, 0.84]  [0.58, 0.74]  [0.68, 0.78]   
AUC           1 Year   Estimate          0.79          0.68          0.73   
                       95% CI    [0.69, 0.88]  [0.57, 0.78]  [0.63, 0.82]   
              2 Year   Estimate          0.77          0.64          0.75   
                       95% CI    [0.68, 0.85]  [0.55, 0.73]  [0.67, 0.83]   
              3 Year   Estimate          0.83          0.66          0.79   
                       95% CI    [0.74, 0.91]  [0.58, 0.75]  [0.71, 0.86]   

Set                                           Test sets combined (treated)  \
Model                                     NAC                          RSF   
Metric Family          Est_CI                                                
C-index       Harrell  Estimate          0.66                         0.75   
                       95% CI    [0.61, 0.71]                   [0.7, 0.8]   
              Uno      Estimate          0.64                         0.76   
                       95% CI     [0.58, 0.7]                 [0.71, 0.81]   
              Antolini Estimate          0.66                         0.74   
                       95% CI    [0.61, 0.71]                  [0.69, 0.8]   
AUC           1 Year   Estimate           0.7                         0.75   
                       95% CI      [0.6, 0.8]                 [0.66, 0.83]   
              2 Year   Estimate          0.68                         0.74   
                       95% CI    [0.59, 0.76]                 [0.66, 0.81]   
              3 Year   Estimate           0.7                         0.81   
                       95% CI    [0.62, 0.77]                 [0.73, 0.87]   

Set                                            
Model                                     NAC  
Metric Family          Est_CI                  
C-index       Harrell  Estimate          0.68  
                       95% CI    [0.62, 0.75]  
              Uno      Estimate           0.7  
                       95% CI    [0.64, 0.75]  
              Antolini Estimate          0.68  
                       95% CI    [0.62, 0.75]  
AUC           1 Year   Estimate          0.71  
                       95% CI      [0.6, 0.8]  
              2 Year   Estimate          0.67  
                       95% CI    [0.58, 0.75]  
             

In [111]:
def get_cal_table(data):
    data = data.copy()
    data = data[data["Model"]=="RSF"]
    # --- metrics ---
    metrics_map = {
        "ici": "ICI",
        "mCalib": "Mean Calibration"
    }
    metrics = {}
    times = ["1y", "2y", "3y"]
    for t in times:
        for metric in metrics_map.keys():
            metrics[f"{metric}_{t}"] = f"{metrics_map[metric]} ({t})"

    id_vars = ["Model"]

    # --- columns to extract and map ---
    value_cols = []
    metric_map = {}

    for split in splits:
        for m, m_name in metrics.items():
            val_col = f"{m}_{split}"
            ci_col = f"CI_{m}_{split}"
            data[ci_col] = data.apply(lambda x: f"{np.round(x[val_col], 2)} {round_ci(x[ci_col])}", axis=1)
            value_cols.extend([val_col, ci_col])
            metric_map[val_col] = m_name
            metric_map[ci_col] = f"{m_name} (95% CI)"

    columns = id_vars + value_cols

    # --- melt ---
    df_long = data[columns].melt(
        id_vars=id_vars,
        value_vars=value_cols,
        var_name="Metric_Source",
        value_name="Value",
    )

    # --- extract Set ---
    pattern = "(" + "|".join(splits) + ")$"
    df_long["Set"] = df_long["Metric_Source"].str.replace("^CI_", "", regex=True).str.extract(pattern)[0]
    df_long["Set"] = df_long["Set"].map(splits).fillna(df_long["Set"])

    # --- map metric names ---
    df_long["Metric"] = df_long["Metric_Source"].map(metric_map)

    # --- ordering ---
    metric_order = [f"Mean Calibration ({t}) (95% CI)" for t in times] + \
                   [f"ICI ({t}) (95% CI)" for t in times]

    df_long["Metric"] = pd.Categorical(df_long["Metric"], categories=metric_order, ordered=True)
    df_long["Set"] = pd.Categorical(
        df_long["Set"],
        categories=list(splits.values()),
        ordered=True,
    )
    # --- pivot ---
    table = df_long.pivot_table(
        index=["Set"],
        columns=["Metric"],
        values="Value",
        aggfunc="first",
        observed=False,
    )
    # current columns
    cols = table.columns.tolist()

    # convert to list of tuples (level0, level1)
    multi_cols = [split_metric_name(c) for c in cols]

    # set MultiIndex
    table.columns = pd.MultiIndex.from_tuples(multi_cols, names=["Metric", "Time"])
    return table

In [112]:
table_cal = get_cal_table(res)
table_cal.to_excel(join(out_folder, "calibration_table.xlsx"))

In [113]:
data = res.copy()
data = data[data["Model"]=="RSF"]

# --- metrics ---

metrics = {}
times = ["1y", "2y", "3y"]
for t in times:
    for metric in metrics_map.keys():
        metrics[f"{metric}_{t}"] = f"{metrics_map[metric]} ({t})"

id_vars = ["Model"]

# --- columns to extract and map ---
value_cols = []
metric_map = {}

for split in splits:
    for m, m_name in metrics.items():
        val_col = f"{m}_{split}"
        ci_col = f"CI_{m}_{split}"
        data[ci_col] = data.apply(lambda x: f"{np.round(x[val_col], 2)}\n{round_ci(x[ci_col])}", axis=1)
        value_cols.extend([ci_col])
        metric_map[val_col] = m_name
        metric_map[ci_col] = f"{m_name}"

columns = id_vars + value_cols

# --- melt ---
df_long = data[columns].melt(
    id_vars=id_vars,
    value_vars=value_cols,
    var_name="Metric_Source",
    value_name="Value",
)

# --- extract Set ---
pattern = "(" + "|".join(splits) + ")$"
df_long["Set"] = df_long["Metric_Source"].str.replace("^CI_", "", regex=True).str.extract(pattern)[0]
df_long["Set"] = df_long["Set"].map(splits).fillna(df_long["Set"])

# --- map metric names ---
df_long["Metric"] = df_long["Metric_Source"].map(metric_map)

# --- ordering ---
metric_order = [f"Mean Calibration ({t})" for t in times] + \
               [f"ICI ({t})" for t in times]

df_long["Metric"] = pd.Categorical(df_long["Metric"], categories=metric_order, ordered=True)
df_long["Set"] = pd.Categorical(
    df_long["Set"],
    categories=list(splits.values()),
    ordered=True,
)
# --- pivot ---
table = df_long.pivot_table(
    index=["Metric"],
    columns=["Set"],
    values="Value",
    aggfunc="first",
    observed=False,
)
# current columns
cols = table.index.tolist()

# convert to list of tuples (level0, level1)
multi_cols = [split_metric_name(c) for c in cols]

# set MultiIndex
table.index = pd.MultiIndex.from_tuples(multi_cols, names=["Metric", "Time"])
table.to_excel(join(out_folder, "calibration_table_v2.xlsx"))

In [114]:
# data = res.copy()
# data = data[data["Model"]=="RSF"]
# splits = {
#     "val": "Internal Validation",
#     "test_combined": "External Test",
#     "test_treated_first_fup": "External Test (treated)",
# }
# # --- metrics ---
# metrics_map = {
#     "ici": "ICI",
#     "mCalib": "Mean Calibration"
# }
# metrics = {}
# times = ["1y", "2y", "3y"]
# for t in times:
#     for metric in metrics_map.keys():
#         metrics[f"{metric}_{t}"] = f"{metrics_map[metric]} ({t})"
#
# id_vars = ["Model"]
#
# # --- columns to extract and map ---
# value_cols = []
# metric_map = {}
#
# for split in splits:
#     for m, m_name in metrics.items():
#         val_col = f"{m}_{split}"
#         ci_col = f"CI_{m}_{split}"
#         data[ci_col] = data.apply(lambda x: f"{np.round(x[val_col], 2)} {round_ci(x[ci_col])}", axis=1)
#         value_cols.extend([ci_col])
#         metric_map[val_col] = m_name
#         metric_map[ci_col] = f"{m_name}"
#
# columns = id_vars + value_cols
#
# # --- melt ---
# df_long = data[columns].melt(
#     id_vars=id_vars,
#     value_vars=value_cols,
#     var_name="Metric_Source",
#     value_name="Value",
# )
#
# # --- extract Set ---
# pattern = "(" + "|".join(splits) + ")$"
# df_long["Set"] = df_long["Metric_Source"].str.replace("^CI_", "", regex=True).str.extract(pattern)[0]
# df_long["Set"] = df_long["Set"].map(splits).fillna(df_long["Set"])
#
# # --- map metric names ---
# df_long["Metric"] = df_long["Metric_Source"].map(metric_map)
#
# # --- ordering ---
# metric_order = [f"Mean Calibration ({t})" for t in times] + \
#                [f"ICI ({t})" for t in times]
#
# df_long["Metric"] = pd.Categorical(df_long["Metric"], categories=metric_order, ordered=True)
# df_long["Set"] = pd.Categorical(
#     df_long["Set"],
#     categories=list(splits.values()),
#     ordered=True,
# )
# # --- pivot ---
# table = df_long.pivot_table(
#     index=["Metric"],
#     columns=["Set"],
#     values="Value",
#     aggfunc="first",
#     observed=False,
# )
# # current columns
# cols = table.index.tolist()
#
# # convert to list of tuples (level0, level1)
# multi_cols = [split_metric_name(c) for c in cols]
#
# # set MultiIndex
# table.index = pd.MultiIndex.from_tuples(multi_cols, names=["Metric", "Time"])
# table.to_excel("calibration_table_v3.xlsx")

In [115]:
# res_recal = pd.read_excel(join(base_path, folder, "results_table_after_recalibration.xlsx"))
# res_recal

In [116]:
# metrics = ["mCalib", "ici"]
# times = ["1y", "2y", "3y"]
# cols = []
# for metric in metrics:
#     for t in times:
#         col = f"{metric}_{t}"
#         col_ci = f"CI_{metric}_{t}"
#         res_recal[col] = res_recal.apply(lambda x: f"{x[col]} {x[col_ci]}", axis=1)
#         cols.append(col)
# cols += ["re-calibrated"]
# res_recal

In [117]:
# # Convert to DataFrame
#
# df = res_recal[cols].copy()
# df["re-calibrated"] = df["re-calibrated"].replace({
#     "Yes": "Recalibrated",
#     "No": "Original",
# })
# df = df.set_index("re-calibrated")
#
# # Make MultiIndex columns
# df.columns = pd.MultiIndex.from_tuples(
#     [tuple(col.split("_")) for col in df.columns],
#     names=["Metric", "Year"]
# )
#
# names_map = {
#     "mCalib": "Mean Calibration",
#     "ici": "ICI",
#     "1y": "1 Year",
#     "2y": "2 Years",
#     "3y": "3 Years",
# }
# new_tuples = [(names_map.get(m, m), names_map.get(y, y)) for m, y in df.columns]
# df.columns = pd.MultiIndex.from_tuples(new_tuples, names=df.columns.names)
#
# df = df.iloc[::-1]
# df.to_excel(join(out_folder, "calibration_table_recalibrated.xlsx"))

In [88]:
from PIL import Image
from os.path import join

# Load images
base_path = "/home/aici/PycharmProjects/survival_analysis/results/attr/ml/20_02_2026_brier_best_hyperparams/plots"
img1 = Image.open(join(base_path, "km_stratified_by_risk_nac_swiss.png"))
img2 = Image.open(join(base_path, "km_stratified_by_risk_nac_vienna.png"))

# Resize second image if widths differ
if img1.width != img2.width:
    new_height = int(img2.height * img1.width / img2.width)
    img2 = img2.resize((img1.width, new_height), Image.LANCZOS)

# Create a new blank image with combined height
combined_height = img1.height + img2.height
combined_image = Image.new("RGB", (img1.width, combined_height), (255, 255, 255))

# Paste images one on top of the other
combined_image.paste(img1, (0, 0))
combined_image.paste(img2, (0, img1.height))

# Save with 600 DPI
combined_image.save(
    join(base_path, "km_stratified_combined.png"),
    dpi=(600, 600)
)

print("Combined image saved as km_stratified_combined.png")

Combined image saved as km_stratified_combined.png


In [94]:
from PIL import Image

# Load images
img1 = Image.open(join(base_path, "calibration_all_sets_1y_SelectKBest_RSF_rand_refit.png"))
img2 = Image.open(join(base_path, "calibration_all_sets_2y_SelectKBest_RSF_rand_refit.png"))
img3 = Image.open(join(base_path, "calibration_all_sets_3y_SelectKBest_RSF_rand_refit.png"))

images = [img1, img2, img3]

# Ensure all images have the same height (resize proportionally if needed)
max_height = max(img.height for img in images)

resized_images = []
for img in images:
    if img.height != max_height:
        new_width = int(img.width * max_height / img.height)
        img = img.resize((new_width, max_height), Image.LANCZOS)
    resized_images.append(img)

# Compute total width
total_width = sum(img.width for img in resized_images)

# Create blank canvas
combined_image = Image.new("RGB", (total_width, max_height), (255, 255, 255))

# Paste images horizontally
x_offset = 0
for img in resized_images:
    combined_image.paste(img, (x_offset, 0))
    x_offset += img.width

# Save with 600 DPI
combined_image.save(
    join(base_path, "calibration_all_sets_combined_horizontal.png"),
    dpi=(600, 600)
)

print("Saved as calibration_all_sets_combined_horizontal.png")


Saved as calibration_all_sets_combined_horizontal.png


In [135]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Load images
img1 = mpimg.imread(join(base_path, f"decision_curve_1y_SelectKBest_RSF_rand_refit_swiss_1y.png"))
img2 = mpimg.imread(join(base_path, f"decision_curve_1y_SelectKBest_RSF_rand_refit_vienna_1y.png"))

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Swiss
axes[0].imshow(img1)
axes[0].set_title("Swiss Test", fontweight="bold", fontsize=20, x=0.55)
axes[0].axis("off")

# Vienna
axes[1].imshow(img2)
axes[1].set_title("Vienna Test", fontweight="bold", fontsize=20, x=0.55)
axes[1].axis("off")

plt.tight_layout()

# Save at 600 dpi
plt.savefig(
    join(base_path, "decision_curve_combined_1y.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.close()

In [139]:
time_names = ["2y", "3y"]
time_names_plot = ["2 years", "3 years"]

# Create figure
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for i in range(len(time_names)):
    t = time_names[i]
    t_plot = time_names_plot[i]

    # Load images
    img1 = mpimg.imread(join(base_path, f"decision_curve_{t}_SelectKBest_RSF_rand_refit_swiss_{t}.png"))
    img2 = mpimg.imread(join(base_path, f"decision_curve_{t}_SelectKBest_RSF_rand_refit_vienna_{t}.png"))

    # Swiss
    axes[i, 0].imshow(img1)
    axes[i, 0].set_title(f"Swiss Test - {t_plot}", fontweight="bold", fontsize=14, x=0.6, pad=2)
    axes[i, 0].axis("off")

    # Vienna
    axes[i, 1].imshow(img2)
    axes[i, 1].set_title(f"Vienna Test - {t_plot}", fontweight="bold", fontsize=14, x=0.6, pad=2)
    axes[i, 1].axis("off")

plt.tight_layout()

# Save at 600 dpi
plt.savefig(
    join(base_path, "decision_curves_combined_2y_3y.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.close()

In [5]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

base_path = "/home/aici/PycharmProjects/survival_analysis/results/attr/ml/20_02_2026_brier_best_hyperparams/plots"

# Load images
img1 = mpimg.imread(join(base_path, "local_xai_highRisk_lowNAC.png"))
img2 = mpimg.imread(join(base_path, "local_xai_lowRisk_highNAC.png"))

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Swiss
axes[0].imshow(img1)
axes[0].set_title("High ML risk - NAC Stage I", fontweight="bold", fontsize=20, x=0.55)
axes[0].axis("off")

# Vienna
axes[1].imshow(img2)
axes[1].set_title("Low ML risk - NAC Stage II", fontweight="bold", fontsize=20, x=0.55)
axes[1].axis("off")

plt.tight_layout()

# Save at 600 dpi
plt.savefig(
    join(base_path, "local_xai_combined.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()
